# Segundo modelo víctima — LendingClub

## 0. Setup

In [ ]:
import sys
from pathlib import Path
def _raiz(i):
    for d in [i, *i.parents]:
        if (d/'src'/'tesis'/'modelo_base.py').exists(): return d
    raise RuntimeError('raiz')
ROOT = _raiz(Path.cwd())
import pandas as pd, numpy as np, warnings, time, pickle
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from datetime import datetime, timezone
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, confusion_matrix, roc_curve)
from scipy.stats import ks_2samp
RUTA = ROOT/'data'/'lendingclub'/'accepted_2007_to_2018Q4.csv'
ROOT

## 1. Variables seleccionadas

In [ ]:
NUM = ['loan_amnt','int_rate','installment','annual_inc','dti','delinq_2yrs','fico_range_low',
       'inq_last_6mths','open_acc','pub_rec','revol_bal','revol_util','total_acc',
       'pub_rec_bankruptcies','mort_acc','emp_length']
CAT = ['term','sub_grade','home_ownership','verification_status','purpose','application_type']
EMP = 'emp_length'
len(NUM), len(CAT)

## 2. Carga y target

In [ ]:
raw = pd.read_csv(RUTA, usecols=['issue_d','loan_status']+[c for c in NUM if c!=EMP]+[EMP]+CAT, low_memory=False)
malos  = ['Charged Off','Default','Does not meet the credit policy. Status:Charged Off']
buenos = ['Fully Paid','Does not meet the credit policy. Status:Fully Paid']
raw['issue_d'] = pd.to_datetime(raw['issue_d'], format='%b-%Y', errors='coerce')
raw['acabado'] = raw['loan_status'].isin(malos+buenos)
raw['anio']    = raw['issue_d'].dt.year
df = raw[raw['acabado']].copy()
df['y'] = df['loan_status'].isin(malos).astype(int)
df.shape, round(df['y'].mean(), 3)

## 3. Ventanas temporales y madurez

In [ ]:
mad = raw.groupby('anio')['acabado'].mean()
tab = pd.DataFrame({'pct_madurado': (mad*100).round(0)})
tab['estado'] = np.where(mad>=0.90,'confiable', np.where(mad>=0.66,'parcial','censurado'))
TRAIN = (df['issue_d']>='2011-01-01') & (df['issue_d']<='2012-12-01')
OOT   = (df['issue_d']>='2013-01-01') & (df['issue_d']<='2013-12-01')
PROD  = (df['issue_d']>='2014-01-01') & (df['issue_d']<='2016-03-01')
tab

## 4. Limpieza

In [ ]:
for c in ['int_rate','revol_util']:
    df[c] = pd.to_numeric(df[c].astype(str).str.replace('%','',regex=False), errors='coerce')
df['term'] = df['term'].astype(str).str.extract(r'(\d+)')
df[EMP] = df[EMP].map({'< 1 year':0,'1 year':1,'2 years':2,'3 years':3,'4 years':4,'5 years':5,
                       '6 years':6,'7 years':7,'8 years':8,'9 years':9,'10+ years':10})
df[['int_rate','revol_util','term',EMP]].dtypes

## 5. Relevancia univariada

In [ ]:
tr = df[TRAIN]
filas = []
for c in NUM:
    a = roc_auc_score(tr['y'], tr[c].fillna(tr[c].median()))
    filas.append((c, max(a, 1-a)))
rel = pd.DataFrame(filas, columns=['variable','auc']).sort_values('auc')
plt.figure(figsize=(8,5))
plt.barh(rel['variable'], rel['auc'], color='#457B9D')
plt.axvline(0.5, color='gray', ls=':'); plt.xlim(0.5, 0.7)
plt.title('AUC univariada — train 2011-12'); plt.tight_layout(); plt.show()
rel.sort_values('auc', ascending=False).round(3).reset_index(drop=True)

## 6. Modelo

In [ ]:
pipe = Pipeline([('prep', ColumnTransformer([
        ('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), NUM),
        ('cat', Pipeline([('imp',SimpleImputer(strategy='most_frequent')),('oh',OneHotEncoder(handle_unknown='ignore'))]), CAT)])),
    ('logit', LogisticRegression(max_iter=1000))])
pipe.fit(df.loc[TRAIN, NUM+CAT], df.loc[TRAIN, 'y'])

## 7. Métricas de clasificación binaria

In [ ]:
def metricas(nombre, mask):
    X, y = df.loc[mask, NUM+CAT], df.loc[mask, 'y']
    p = pipe.predict_proba(X)[:, 1]; pred = (p>=0.5).astype(int); a = roc_auc_score(y, p)
    return {'muestra':nombre,'n':len(y),'AUC':round(a,3),'Gini':round(2*a-1,3),
            'KS':round(ks_2samp(p[y==1],p[y==0]).statistic,3),
            'Accuracy':round(accuracy_score(y,pred),3),'Precision':round(precision_score(y,pred,zero_division=0),3),
            'Recall':round(recall_score(y,pred,zero_division=0),3),'F1':round(f1_score(y,pred,zero_division=0),3)}
pd.DataFrame([metricas('TRAIN 2011-12',TRAIN), metricas('OOT 2013',OOT), metricas('PROD 2014-16Q1',PROD)])

## 8. Curvas ROC y KS

In [ ]:
Xo, yo = df.loc[OOT, NUM+CAT], df.loc[OOT, 'y']
po = pipe.predict_proba(Xo)[:, 1]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fpr, tpr, _ = roc_curve(yo, po)
ax1.plot(fpr, tpr, color='#1D3557', label=f'AUC={roc_auc_score(yo,po):.3f}')
ax1.plot([0,1],[0,1],'--',color='gray'); ax1.set_title('ROC — OOT 2013')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR'); ax1.legend()
orden = np.argsort(po); yb = (yo.to_numpy()[orden]==0); ym = (yo.to_numpy()[orden]==1)
cb = np.cumsum(yb)/yb.sum(); cm = np.cumsum(ym)/ym.sum()
eje = np.arange(1,len(po)+1)/len(po); i = np.argmax(np.abs(cm-cb))
ax2.plot(eje, cb, color='#457B9D', label='buenos'); ax2.plot(eje, cm, color='#B0302B', label='malos')
ax2.vlines(eje[i], cb[i], cm[i], color='black', label=f'KS={abs(cm-cb)[i]:.3f}')
ax2.set_title('KS — OOT 2013'); ax2.set_xlabel('poblacion ordenada'); ax2.legend()
plt.tight_layout(); plt.show()

## 9. Performance evolutivo

In [ ]:
df['p'] = pipe.predict_proba(df[NUM+CAT])[:, 1]
df['q'] = df['issue_d'].dt.to_period('Q')
ev = df[df['issue_d']<='2016-03-01'].groupby('q').apply(
    lambda d: pd.Series({'n':len(d),'tasa_malos':d['y'].mean(),
                         'AUC':roc_auc_score(d['y'],d['p']) if d['y'].nunique()==2 else np.nan}))
x = range(len(ev)); labs = [str(q) for q in ev.index]
fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(x, ev['AUC'], 'o-', color='#1D3557', label='AUC')
def band(q0, q1, color, txt):
    i0 = [i for i,q in enumerate(ev.index) if str(q)>=q0][0]
    i1 = [i for i,q in enumerate(ev.index) if str(q)<=q1][-1]
    ax.axvspan(i0-0.5, i1+0.5, color=color, alpha=0.12); ax.text((i0+i1)/2, 0.74, txt, ha='center', fontsize=9, color=color)
band('2011Q1','2012Q4','#2A7F62','train'); band('2013Q1','2013Q4','#C8861E','OOT'); band('2014Q1','2016Q1','#457B9D','produccion')
ax.set_xticks(list(x)); ax.set_xticklabels(labs, rotation=90, fontsize=8)
ax.set_ylabel('AUC'); ax.set_title('AUC por trimestre'); ax.grid(alpha=0.3); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()
ev.round(3)

## 10. Guardar y congelar el modelo

In [ ]:
auc_oot = roc_auc_score(df.loc[OOT,'y'], pipe.predict_proba(df.loc[OOT, NUM+CAT])[:,1])
artefacto = {
    'pipeline': pipe,
    'num': NUM, 'cat': CAT, 'features': NUM+CAT,
    'metadata': {
        'modelo': 'LendingClub default — regresion logistica',
        'rol': 'segundo modelo victima',
        'train': '2011-01..2012-12', 'oot': '2013', 'produccion': '2014-01..2016-03',
        'corte_madurez': '2016-03',
        'auc_oot': round(float(auc_oot), 4),
        'n_train': int(TRAIN.sum()),
        'versiones': {'sklearn': sklearn.__version__, 'pandas': pd.__version__},
        'congelado_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    },
}
ruta = ROOT/'models'/'modelo_lendingclub.pkl'
ruta.parent.mkdir(parents=True, exist_ok=True)
with open(ruta, 'wb') as f:
    pickle.dump(artefacto, f)
ruta

In [ ]:
with open(ROOT/'models'/'modelo_lendingclub.pkl', 'rb') as f:
    art = pickle.load(f)
art['metadata']